# 06 – Deal Detection

The final model predicts one price per car. To decide whether a listing is a *deal*, I need a
**range** of fair prices, since the model is much more certain about some cars than others.
A deal is a car listed below the bottom of its own range.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = (2023 - df["year"]).clip(lower=0)
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [2]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [3]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)

### Three models: low, middle and high
- `mid`: the normal model (most likely price)
- `low`: quantile regression at the 10th percentile, the price only 10% of similar cars sell below
- `high`: the 90th percentile

Together they give every car a fair-price range. Models are saved to `models/` so they only train once.

In [4]:
import os, joblib
import lightgbm as lgb

os.makedirs("../models", exist_ok=True)

def train_lgbm(name, **extra):
    path = f"../models/{name}.joblib"
    if os.path.exists(path):
        print(f"Loaded {name}")
        return joblib.load(path)
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                          min_child_samples=20, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=42, verbose=-1, **extra)
    m.fit(train_fit[features], np.log(train_fit["price"]),
          eval_set=[(valid[features], np.log(valid["price"]))],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    joblib.dump(m, path)
    return m

mid  = train_lgbm("lgbm_mid")
low  = train_lgbm("lgbm_q10", objective="quantile", alpha=0.10)
high = train_lgbm("lgbm_q90", objective="quantile", alpha=0.90)

Loaded lgbm_mid
Loaded lgbm_q10
Loaded lgbm_q90


### Calibrating the ranges
The raw quantile ranges were slightly too narrow: only 75% of test prices fell inside, against an
80% target. Using conformalized quantile regression, every range is widened by the amount needed
to reach 80% coverage on the validation set.
*(The validation set was also used for early stopping, so it isn't perfectly independent, but the effect is small.)*

In [5]:
yv = np.log(valid["price"])
lo_v, hi_v = low.predict(valid[features]), high.predict(valid[features])
scores = np.maximum(lo_v - yv, yv - hi_v)
q = np.quantile(scores, 0.80)

In [6]:
X = test[features]
test = test.assign(
    pred_mid=np.exp(mid.predict(X)),
    pred_low=np.exp(low.predict(X) - q),
    pred_high=np.exp(high.predict(X) + q),
)
# Make sure low <= mid <= high
test["pred_low"] = test[["pred_low", "pred_mid"]].min(axis=1)
test["pred_high"] = test[["pred_high", "pred_mid"]].max(axis=1)

In [7]:
inside = test["price"].between(test["pred_low"], test["pred_high"]).mean()
below = (test["price"] < test["pred_low"]).mean()
above = (test["price"] > test["pred_high"]).mean()
print(f"Inside range: {inside:.1%}  (target ~80%)")
print(f"Below range:  {below:.1%}  (target ~10%)")
print(f"Above range:  {above:.1%}  (target ~10%)")

test["range_width_pct"] = (test["pred_high"] - test["pred_low"]) / test["pred_mid"]
bands = pd.cut(test["price"], [0, 10_000, 20_000, 35_000, 60_000, 200_000])
test.groupby(bands, observed=True)["range_width_pct"].median().round(3)

Inside range: 79.9%  (target ~80%)
Below range:  10.1%  (target ~10%)
Above range:  10.0%  (target ~10%)


price
(0, 10000]         0.488
(10000, 20000]     0.264
(20000, 35000]     0.196
(35000, 60000]     0.174
(60000, 200000]    0.171
Name: range_width_pct, dtype: float64

### Are the ranges honest?
After calibration: **79.9% inside, 10.1% below, 10.0% above**, almost exactly on target.

Range width adapts to uncertainty: about 49% of the expected price for cars under $10k, versus
about 17% for cars over $35k. So the deal threshold automatically adapts to how reliable each prediction is.

### Raw flags
Every car priced below the bottom of its range is flagged. Only test-set cars are scored, since the
model has partly "memorized" the training cars and would rarely flag them.

In [8]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


The largest raw discounts are mostly **not** real deals: classic cars (where condition can't be
observed), suspicious placeholder prices, and accident-damaged vehicles. They need separating out.

$2,500 appeared repeatedly among the most extreme flags. Checking which sellers use it shows whether
it's a placeholder price (e.g. a down payment or "call for price") rather than a real asking price.

In [9]:
df.loc[df["price"] == 2500, "seller_name"].value_counts().head(10)

seller_name
Car Collection Inc                   7
Suburban Ford of Sterling Heights    2
Corwin Public Wholesale              2
REM Motors LLC                       2
Central FL Motors LLC                2
Shea Buick GMC                       1
Mike Molstead Motors                 1
Starfire Auto Inc.                   1
Putnam Auto Sales Inc                1
A2Z Auto Group                       1
Name: count, dtype: int64

### Classifying flags
Each flagged car gets one label, checked in this order:
1. **Classic**: model year before 2000 (out of scope, since condition drives these prices)
2. **Likely data error**: more than 50% below expected
3. **Check mileage**: 5+ years old and under 1,500 miles a year (possible typo)
4. **Accident history**: an accident was reported
5. **Possible deal**: everything else

In [10]:
def classify(row):
    if row["year"] < 2000:
        return "Classic (out of scope)"
    if row["discount"] > 0.5:
        return "Likely data error"
    if row["age"] >= 5 and row["miles_per_year"] < 1500:
        return "Check mileage"
    if row["accidents_or_damage"] == "Yes":
        return "Accident history"
    return "Possible deal"

deals["flag_type"] = deals.apply(classify, axis=1)
deals["flag_type"].value_counts()

flag_type
Possible deal             11196
Accident history           3647
Classic (out of scope)      131
Likely data error           125
Check mileage                38
Name: count, dtype: int64

### Deal score and tiers
Sorting by % discount favours cheap old cars, whose ranges are very wide. Instead:

**deal score = (bottom of range − price) / range width**

Being below a narrow, confident range counts for more than being below a wide, uncertain one.
Range width has a minimum of 10% of the expected price: occasionally the three models cross, and
correcting the order collapses one side of the range, which would otherwise inflate the score.

| Tier | Score | Meaning |
|---|---|---|
| Slight | < 0.25 | Just below range, likely normal variation |
| Good | 0.25–0.5 | Clearly below similar cars |
| Strong | 0.5–1.0 | Rare: about 0.6% of all listings |
| Review | > 1.0 | Too far below to trust without checking |

In [11]:
width = (deals["pred_high"] - deals["pred_low"]).clip(lower=0.10 * deals["pred_mid"])
deals["deal_score"] = (deals["pred_low"] - deals["price"]) / width
deals["savings"] = deals["pred_mid"] - deals["price"]

real = deals[deals["flag_type"] == "Possible deal"].copy()
print(real["deal_score"].describe(percentiles=[.5, .75, .9, .95, .99]).round(3))

real["tier"] = pd.cut(real["deal_score"], bins=[0, 0.25, 0.5, 1.0, np.inf],
                      labels=["Slight", "Good", "Strong", "Review"],
                      include_lowest=True)
real["tier"].value_counts()

count    11196.000
mean         0.206
std          0.224
min          0.000
50%          0.137
75%          0.277
90%          0.479
95%          0.636
99%          1.091
max          2.786
Name: deal_score, dtype: float64


tier
Slight    8007
Good      2182
Strong     867
Review     140
Name: count, dtype: int64

In [12]:
show = ["manufacturer", "model", "year", "mileage", "price", "pred_low",
        "pred_mid", "pred_high", "savings", "deal_score", "tier"]
strong = real[real["tier"] == "Strong"].sort_values("deal_score", ascending=False)
strong[show].head(20).round(2)

,manufacturer,model,year,mileage,price,pred_low,pred_mid,pred_high,savings,deal_score,tier
729412,Volkswagen,Golf R 4-Door,2016,80085.0,21495.0,24709.88,27936.36,27936.36,6441.36,1.00,Strong
471021,Lexus,ES 250,2021,13565.0,33159.0,36952.94,38144.25,40212.54,4985.25,0.99,Strong
342913,Hyundai,Azera Limited,2017,32441.0,17995.0,21700.19,23586.60,25426.94,5591.60,0.99,Strong
688712,Toyota,Corolla LE,2021,40130.0,15995.0,20395.11,22514.62,24843.92,6519.62,0.99,Strong
142050,Chevrolet,Colorado LT,2020,6855.0,27995.0,33048.97,36078.50,38165.11,8083.50,0.99,Strong
112902,Chevrolet,Spark LS,2022,289.0,15750.0,18587.34,19424.36,21465.27,3674.36,0.99,Strong
686089,Toyota,Sienna XLE,2017,42438.0,22500.0,30594.08,32442.30,38855.30,9942.30,0.98,Strong
307607,Honda,HR-V EX,2020,23579.0,20995.0,24640.98,25986.67,28365.38,4991.67,0.98,Strong
474032,Lexus,LS 460 Base,2010,67600.0,13990.0,18017.75,20166.96,22139.31,6176.96,0.98,Strong
1359,Acura,RLX Advance Package,2017,21041.0,28990.0,32279.86,33781.69,35077.41,4791.69,0.97,Strong


In [13]:
summary = real.groupby("tier", observed=True).agg(
    cars=("price", "size"),
    median_savings=("savings", "median"),
    pct_price_dropped=("had_price_drop", "mean"),
)
print(summary.round(2))
print(f"\nAll test listings price-dropped: {test['had_price_drop'].mean():.1%}")

        cars  median_savings  pct_price_dropped
tier                                           
Slight  8007         2898.88               0.62
Good    2182         3932.26               0.62
Strong   867         4978.92               0.55
Review   140         6852.34               0.46

All test listings price-dropped: 54.2%


Modest deals (Slight/Good) are disproportionately listings that had already been price-reduced (62% vs 54% overall). The strongest discounts are not: Strong deals match the overall rate (55%) and Review cases fall below it (46%), meaning they were listed low from the start. Large discounts are therefore driven by initial pricing, which may reflect genuine bargains or undisclosed issues. This supports flagging extreme cases for review rather than ranking them as the best deals.

In [14]:
strong.sample(15, random_state=42)[show].round(2)

,manufacturer,model,year,mileage,price,pred_low,pred_mid,pred_high,savings,deal_score,tier
350549,Hyundai,Santa Fe Limited Ultimate,2018,33775.0,23778.0,27539.27,30352.10,33187.62,6574.10,0.67,Strong
696004,Toyota,Prius LE,2020,28000.0,21995.0,25382.32,27524.85,30719.17,5529.85,0.63,Strong
501570,Mazda,CX-5 Touring,2021,55383.0,18495.0,22398.42,24407.66,26820.60,5912.66,0.88,Strong
59401,BMW,530 i,2022,10955.0,42903.0,47328.88,51231.58,55777.48,8328.58,0.52,Strong
704462,Toyota,Tacoma TRD Sport,2021,9512.0,33497.0,37041.38,39396.27,42510.72,5899.27,0.65,Strong
411771,Kia,Optima Hybrid EX,2016,94293.0,12500.0,13635.48,14498.82,15639.81,1998.82,0.57,Strong
691951,Toyota,Corolla XLE,2018,131952.0,11899.0,13921.28,15466.70,17924.75,3567.70,0.51,Strong
676402,Toyota,Sequoia SR5,2022,14318.0,49998.0,58232.12,64085.41,72757.78,14087.41,0.57,Strong
573840,Nissan,Juke S,2013,107634.0,4950.0,8136.08,9593.31,11480.17,4643.31,0.95,Strong
705241,Toyota,Tacoma Base,2014,54960.0,18595.0,22577.71,25897.53,30290.23,7302.53,0.52,Strong


Strong deals are predominantly 2017–2022 mainstream vehicles listed 13–30% below their expected price, savings of roughly $3k–$11k. The model can't observe condition, title status or undisclosed damage, so flagged listings are leads to investigate, not guaranteed bargains.